In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, date_format, regexp_extract

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA silver")

In [0]:
df = spark.table("bronze.ons_awe")

df_silver = (
    df
    .filter(regexp_extract(col("time_period"), r"^\d{4} [A-Z]{3}$", 0) != "")
    .withColumn("avg_weekly_earnings_gbp", col("avg_weekly_earnings_gbp").cast("double"))
    .withColumn("report_date", to_date(col("time_period"), "yyyy MMM"))
    .withColumn("year_month", date_format(col("report_date").cast("timestamp"), "yyyy-MM"))
    .select("report_date", "year_month", "avg_weekly_earnings_gbp")
)

In [0]:
df_silver.show(5)
df_silver.printSchema()
from pyspark.sql.functions import min, max
df_silver.select(min("report_date"), max("report_date")).show()
print(f"Row count: {df_silver.count()}")

In [0]:
# bronze.ons_awe loaded
# Source: ONS AWE time series KAB9
# 444 rows, mixed granularity (annual, quarterly, monthly)
# Metadata rows skipped at read time
# Filtering to monthly rows happens in silver